In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl evaluate accelerate peft bitsandbytes sentencepiece

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-tu224ou0/unsloth_8a534cafb2344a5abca7f9b9d04775e0
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-tu224ou0/unsloth_8a534cafb2344a5abca7f9b9d04775e0
  Resolved https://github.com/unslothai/unsloth.git to commit b30e2b4b15cb58a9ac09fe466a0be6ab5e162a74
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.1 MB/s eta 0:00:00


In [2]:
import kagglehub
import os
import json
import glob
import gc
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig 

gc.collect()
torch.cuda.empty_cache()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
DATA_FOLDER = "/kaggle/input/datasets/adhamashraf202200953/technical-parsed-questions"
OUTPUT_DIR = "./outputs/UNSLOTH_MCQ_DEEPSEEK_8B"
FINAL_ADAPTER_PATH = "./UNSLOTH_MCQ_DEEPSEEK_8B_LORA"

model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B"
max_seq_length = 2048
dtype = None         
load_in_4bit = True   

csv_files = glob.glob(os.path.join(DATA_FOLDER, "*_mcq.csv"))
if not csv_files:
    raise FileNotFoundError(f"No MCQ CSV files found in {DATA_FOLDER}")

required_columns = [
    "context", "scenario_context", "question",
    "choices/0", "choices/1", "choices/2", "choices/3",
    "correct_choice"
]

all_dfs = []
for file_path in csv_files:
    try:
        df_peek = pd.read_csv(file_path, nrows=2)
        if not all(col in df_peek.columns for col in required_columns):
            print(f"Skipping corrupt file: {os.path.basename(file_path)}")
            continue

        df_clean = pd.read_csv(file_path, usecols=required_columns)
        all_dfs.append(df_clean)
        print(f"Successfully loaded {os.path.basename(file_path)} ({len(df_clean)} rows)")
    except Exception as e:
        print(f"Error reading {os.path.basename(file_path)}: {e}")

if not all_dfs:
    raise ValueError("No valid MCQ files could be loaded.")

combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df = combined_df.dropna(subset=["context", "question", "correct_choice", "choices/0"])

dataset = Dataset.from_pandas(combined_df)
print(f"\nFinal unified dataset size: {len(dataset)} samples.")

Successfully loaded ML_DS_Abstract_Theory_mcq.csv (715 rows)
Successfully loaded DSA_Applied_Enginering_mcq.csv (732 rows)
Successfully loaded SWE_Abstract_Theory_mcq.csv (761 rows)
Skipping corrupt file: DSA_Abstract_Theory_mcq.csv

Final unified dataset size: 2208 samples.


In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

deepseek_prompt_style = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>

You are an expert technical interviewer. Based on the given textbook context, generate a professional workplace scenario context, an analytical interview question, and 4 logical choices detailing the correct answer identifier.

Textbook Context:
{CONTEXT}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

<think>
Creating an advanced technical question based on the textbook context.
</think>Scenario: {SCENARIO}
Question: {QUESTION}
Choices:
A) {CHOICE_0}
B) {CHOICE_1}
C) {CHOICE_2}
D) {CHOICE_3}
Answer: {CORRECT_CHOICE}<|eot_id|>"""

def formatting_prompts_func(examples):
    contexts = examples["context"]
    scenarios = examples["scenario_context"]
    questions = examples["question"]
    c0 = examples["choices/0"]
    c1 = examples["choices/1"]
    c2 = examples["choices/2"]
    c3 = examples["choices/3"]
    answers = examples["correct_choice"]
    
    texts = []
    for c, s, q, o0, o1, o2, o3, a in zip(contexts, scenarios, questions, c0, c1, c2, c3, answers):
        text = deepseek_prompt_style.format(
            CONTEXT=str(c).strip(),
            SCENARIO=str(s).strip(),
            QUESTION=str(q).strip(),
            CHOICE_0=str(o0).strip(),
            CHOICE_1=str(o1).strip(),
            CHOICE_2=str(o2).strip(),
            CHOICE_3=str(o3).strip(),
            CORRECT_CHOICE=str(a).strip()
        )
        texts.append(text)
    return { "text" : texts }

formatted_dataset = dataset.map(formatting_prompts_func, batched = True)
split_dataset = formatted_dataset.train_test_split(test_size=0.05, seed=42)

==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/deepseek-r1-distill-llama-8b-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.6.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Map:   0%|          | 0/2208 [00:00<?, ? examples/s]

In [5]:
training_config = SFTConfig(
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, 
    per_device_train_batch_size = 2, 
    per_device_eval_batch_size = 2,
    gradient_accumulation_steps = 8, 
    warmup_ratio = 0.05,
    num_train_epochs = 3,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    eval_strategy = "steps",
    eval_steps = 25,
    optim = "adamw_8bit", 
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 42,
    output_dir = OUTPUT_DIR,
    report_to = "none",
    save_strategy = "steps",
    save_steps = 25,
    save_total_limit = 1,
    gradient_checkpointing = True
)

if hasattr(training_config, "push_to_hub_token"):
    delattr(training_config, "push_to_hub_token")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = split_dataset["train"],
    eval_dataset = split_dataset["test"],
    args = training_config, 
)

print("\nBooting Unsloth Training engine for DeepSeek-R1...")
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2097 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/111 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.

Booting Unsloth Training engine for DeepSeek-R1...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,097 | Num Epochs = 3 | Total steps = 396
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
25,1.965295,1.897506
50,1.708697,1.714581
75,1.643336,1.649415
100,1.628101,1.605723
125,1.551102,1.577290
150,1.402059,1.575694
175,1.427221,1.557752
200,1.393469,1.544118
225,1.392160,1.528781
250,1.374882,1.518641


Unsloth: Restored added_tokens_decoder metadata in ./outputs/UNSLOTH_MCQ_DEEPSEEK_8B/checkpoint-25/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/UNSLOTH_MCQ_DEEPSEEK_8B/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/UNSLOTH_MCQ_DEEPSEEK_8B/checkpoint-75/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/UNSLOTH_MCQ_DEEPSEEK_8B/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/UNSLOTH_MCQ_DEEPSEEK_8B/checkpoint-125/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/UNSLOTH_MCQ_DEEPSEEK_8B/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/UNSLOTH_MCQ_DEEPSEEK_8B/checkpoint-175/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/UNSLOTH_MCQ_DEEPSEEK_8B/checkpoint-200/tokenizer_config.json.
Unsloth: Restored a

TrainOutput(global_step=396, training_loss=1.4806988931665517, metrics={'train_runtime': 12937.1943, 'train_samples_per_second': 0.486, 'train_steps_per_second': 0.031, 'total_flos': 1.4951505645726106e+17, 'train_loss': 1.4806988931665517, 'epoch': 3.0})

In [6]:
model.save_pretrained(FINAL_ADAPTER_PATH)
tokenizer.save_pretrained(FINAL_ADAPTER_PATH)
print(f"LoRA adapters successfully exported to: {FINAL_ADAPTER_PATH}")

os.system(f'zip -r "./UNSLOTH-DEEPSEEK-8B-LORA.zip" "{FINAL_ADAPTER_PATH}"')
print("Artifact zipped and ready for export.")

Unsloth: Restored added_tokens_decoder metadata in ./UNSLOTH_MCQ_DEEPSEEK_8B_LORA/tokenizer_config.json.


LoRA adapters successfully exported to: ./UNSLOTH_MCQ_DEEPSEEK_8B_LORA
  adding: UNSLOTH_MCQ_DEEPSEEK_8B_LORA/ (stored 0%)
  adding: UNSLOTH_MCQ_DEEPSEEK_8B_LORA/tokenizer.json (deflated 85%)
  adding: UNSLOTH_MCQ_DEEPSEEK_8B_LORA/adapter_config.json (deflated 57%)
  adding: UNSLOTH_MCQ_DEEPSEEK_8B_LORA/README.md (deflated 65%)
  adding: UNSLOTH_MCQ_DEEPSEEK_8B_LORA/chat_template.jinja (deflated 75%)
  adding: UNSLOTH_MCQ_DEEPSEEK_8B_LORA/tokenizer_config.json (deflated 96%)
  adding: UNSLOTH_MCQ_DEEPSEEK_8B_LORA/adapter_model.safetensors (deflated 7%)
Artifact zipped and ready for export.
